# DAY 03 - System: Perplexity (RAG + Agent Hybrid)

## 1. The Problem
- Search engines return links, not answers - user still does the synthesis work.
- LLM alone: stale knowledge (training cutoff) + hallucination risk.
- Perplexity = retrieval (freshness) + generation (synthesis) + citation (verifiability), fused together.

## 2. Architecture (Request Lifecycle)

| Stage | What happens | Confidence |
|--|--|--|
| 1. Query rewriting | Classify intent (time-sensitive vs evergreen), rewrite query into search-optimized form | Confirmed direction; exact mechanism not public |
| 2. Hybrid retrieval | Dense (semantic/vector) + Sparse (keyword/BM25) search combined. Infra = Vespa | Confirmed |
| 3. Rerank | Narrow candidates down to top-k for context window | Plausible, not fully confirmed |
| 4. Planning/agent loop | Only in Pro Search/Deep Research - breaks query into sub-questions, multi-hop retrieval | Confirmed direction |
| 5. Model routing | Own model (Sonar) + partner models (GPT, Claude, Gemini) routed per query | Confirmed |
| 6. Grounded generation | Model constrained to only use retrieved passages + cite each claim | Confirmed direction |
| 7. Output | Answer + inline citations + source list | Confirmed |

**Key correction to remember:** Index is pre-built (crawled + embedded continuously in background). Query time = retrieval only, NOT real-time embedding of the whole web.

## 3. Hard Decisions (Tradeoffs)
- **Latency vs quality**: single-pass retrieval (fast, standard search) vs multi-hop agent loop (slow, Deep Research)
- **Hybrid retrieval cost vs recall**: running dense+sparse+rerank costs more compute, but retrieval quality is the real bottleneck - a bad retrieval can't be fixed downstream
- **Model routing (cost vs capability)**: cheap in-house model (Sonar) for common queries, frontier models reserved for harder cases
- **Autonomy vs control**: agent loop needs step limits/timeouts, or cost & latency become unpredictable

## 4. Failure Modes
- **Hallucination**: reduced (not eliminated) by grounding - model can still mis-cite a source that doesn't actually support the claim
- **Ambiguous intent**: misclassified query → routed to wrong index silently, no error thrown
- **Retrieval/tool failure**: source down/blocked → needs graceful fallback (exact mechanism not public)
- **Adversarial input**: prompt injection via crawled web content - open problem across all RAG systems, not solved by anyone

## 5. Debugging a Multi-Stage Pipeline (General Principle)
When authoritative source is missing from top-5 output:
1. Check raw retrieval output (top-50, unranked) - is the source even retrieved?
   - Not there → problem is in retrieval (query embedding or index), not reranker
   - There but not in top-5 → reranker is suspect
2. Compare reranker's input vs output - look at actual scores given
3. A/B test: bypass reranker, see if raw retrieval score alone surfaces the source
- **Core lesson**: inspect input/output at every stage individually - don't guess from end-to-end output alone

## 6. Web-Scale Crawling & Indexing (the hardest engineering part)

**Not everything is crawled/embedded equally** - an ML model decides indexing priority based on:
- Importance of the URL
- Likely update frequency

**Core tension: Completeness vs Freshness**
- Fixed indexing budget → refreshing old pages competes with indexing new pages
- Solved via: (a) scaling compute, (b) ML-driven prioritization of what to (re)crawl

**Confirmed scale (from Perplexity's own research blog):**
- Exabyte-scale index
- Tens of thousands of CPUs, hundreds of TBs of RAM (crawling+indexing fleet)
- Cold + warm storage tiering (frequently accessed = fast/expensive storage; rare = cheap/cold storage)
- ~200M daily queries (search API)

**Deletion/staleness handling**: not publicly documented. Plausible (unconfirmed) mechanisms: TTL-based re-crawl scheduling, dead-link (404) detection removing entries. Known limitation: stale/deleted content can still surface until next re-crawl - true of all crawl-based search engines, not unique to Perplexity.

**Balanced view**: One external source argues Perplexity's own crawler (PerplexityBot) may be supplementary, and real-time answers may partly lean on established third-party search infra - this is a disputed claim, not officially confirmed.

## 7. Vector DB Internals - How Scale Is Actually Managed

**Basic unit stored:** `{doc_id, text_chunk, embedding_vector, metadata}`

**The scale problem:** 1 embedding (1536 dims, float32) ≈ 6 KB. At ~10 billion chunks → ~60 petabytes just for embeddings (text separate).

**Four levers used to manage this:**

### A) Quantization (memory/compute lever)
- Compress embedding precision: float32 (4 bytes/dim) → int8 (1 byte/dim, 4x smaller) or binary (1 bit/dim, 32x smaller)
- **Both quantized AND full-precision versions are stored** (not a replacement) - quantized version enables a fast rough pass, full-precision enables accurate final scoring
- **Key correction:** the main saving is NOT total storage (storing both can slightly increase disk usage) - the main saving is **RAM usage and compute**:
  - Disk/SSD = cheap, abundant → store both versions here without concern
  - RAM = expensive, limited → only load the small quantized vectors into RAM for the first fast pass across billions; only load full-precision vectors for the tiny shortlist (~1000 candidates) for the final precise pass

### B) ANN - Approximate Nearest Neighbor (speed lever)
Brute-force linear scan across billions of vectors per query is infeasible (O(n)). Two main approaches:

- **IVF (Inverted File Index)**: pre-cluster all vectors (e.g., via k-means) into groups; at query time, only search within the closest few cluster(s) instead of everything
- **HNSW (Hierarchical Navigable Small World)**: multi-layer graph of connected similar vectors; search starts at sparse top layer (long jumps), descends layer by layer refining position - like a highway-to-local-street navigation
- **Both are greedy/approximate**: can lock into a local optimum instead of true global nearest neighbor → trades some accuracy for massive speed gain (~O(log n) instead of O(n))

### C) Sharding
Data split across many machines (by hash/topic); query searches relevant shards in parallel, results merged. Needed because no single machine can hold it all.

### D) Tiered storage
Hot/warm (fast storage) for frequently accessed/fresh content; cold (cheap storage) for rare/old content.

**Full retrieval funnel, combining everything:**
```
10 billion vectors (quantized, small enough for RAM)
   ↓ ANN search (HNSW/IVF) - fast, approximate
~1,000 candidates
   ↓ Pull their full-precision vectors into RAM (small data, ~6MB)
   ↓ Precise similarity scoring
Top 5 → sent to reranker → final context → LLM generation
```

**Core takeaway:** Quantization and ANN are separate but complementary levers - quantization controls *how much data* you need to move into RAM, ANN controls *how much of the dataset* you even need to look at. Together they make web-scale vector search computationally feasible without needing to "load everything."

--
**Cross-question answered:** Perplexity rewrites the query before searching because raw conversational queries are poorly optimized for retrieval - the rewrite step classifies intent (time-sensitive vs evergreen) and reformulates the query into a form that matches how the index actually organizes and retrieves content.

# Perplexity Query Rewriting: Key Concepts

## Why Raw Queries Are Not Retrieval Friendly

A raw conversational query like "why does my react app keep re-rendering" is written like a question, not like an answer. Search indexes are built from document style content, so a first person question does not match well with how content is actually written and stored.

## HyDE (Hypothetical Document Embeddings)

Instead of embedding the raw question, the LLM writes a hypothetical answer that reads like a real document. This synthetic passage is then embedded and used for retrieval, since dense retrievers perform better on document to document similarity than question to document similarity.

Example: "why does my react app keep re-rendering" becomes something like "React re-renders occur when state or props change, or when a parent component re-renders."

### Risk of HyDE

If the LLM hallucinates in the hypothetical document, the retrieval becomes poor. A wrong guess in the hypothetical answer leads the retriever to fetch documents that are semantically similar to the wrong concept, not the actually correct one. This is sometimes called hallucination snowballing, since the retrieval step reinforces the original error instead of correcting it.

### Mitigation

Use a combination of the raw query embedding and one or more hypothetical document embeddings. Retrieve using both, then merge and re-rank the results (often using Reciprocal Rank Fusion, RRF). This creates a safety net: if the hypothetical document goes in the wrong direction, the raw query can still surface the correct documents, and the re-ranker picks the best results overall.

## Multi Query Reformulation

Instead of generating one optimized query, the system can generate multiple reformulations to cover a broader aspect of the topic. This gives richer and more complete retrieval results.

Tradeoff: more reformulations mean more retrieval calls, which increases cost and latency. There are also diminishing returns, since later reformulations often return documents similar to the ones already retrieved by earlier ones. In practice, systems limit this to a small number of reformulations, such as two or three.

## Intent Classification: Evergreen vs Time Sensitive

Before retrieval, the query is classified as either evergreen (stable, unchanging information) or time sensitive (recent or live information).

- Evergreen queries, such as "what is gradient descent," can be served from a static or cached index with pre crawled and pre embedded content.
- Time sensitive queries, such as "who won today's F1 race," require a fresh crawl or live data source, since a static index would be outdated.

### Primary Purpose of This Classification

This step is not just about saving cost and reducing latency. It is a correctness critical step. If a time sensitive query is incorrectly routed to a static or evergreen index, the answer will not just be slow, it will be factually wrong, since the static index does not contain the latest information. So intent classification protects both efficiency and correctness, with correctness being the more critical concern.

## Full Pipeline Summary

Query, then Intent Classification (time sensitive or evergreen), then Reformulation (raw query plus HyDE and multi query), then Retrieval (across sources), then Fusion or Re ranking, then Answer generation.